# **PLEASE MAKE YOUR OWN COPY OF THIS NOTEBOOK**

# CIS 5450 Homework 2: SQL (Spring 2026)

**Due: Thursday, February 26nd, 10:00 PM EST**
**Total Points: 124**

Welcome to Homework 2! By now, you should be familiar with the world of data science and the Pandas library. This assignment focuses on helping you get to grips with a new tool: SQL. Through this homework, we will be working with SQL and DuckDB by exploring an [Indego](https://www.rideindego.com/about/data/) dataset containing information on bikeshare trips taken in Philadelphia.


We **strongly** encourage you to review the slides/material as you work through this assignment:- make a personal inventory of SQL clauses and what they do
- draw a diagram that helps you remember the differences between different kinds of joins- etc.

**Before you begin:**- Be sure to click "Copy to Drive" to make sure you're working on your own personal version of the homework- Check the **pinned FAQ post** on Ed for updates! If you have been stuck, chances are other students have also faced similar problems.

## Part 0: Libraries and Set Up Jargon (The usual wall of imports)

Run but do *not* edit these cells.

In [ ]:
%set_env HW_ID=cis5450_spr26_HW2

In [ ]:

%%capture
!pip install penngrader-client

In [ ]:
!pip install sqlalchemy==1.4.46
!pip install pandasql
!pip install geopy
!pip install -U kaleido
!pip3 install duckdb

In [ ]:
from penngrader.grader import *
import pandas as pd
import datetime as dt
import geopy.distance as gp
import matplotlib.image as mpimg
import plotly.express as px
import duckdb

from matplotlib.dates import date2num
import matplotlib.pyplot as plt
import math
import re
import json
import os
from collections import Counter
import random

### Setting up PennGrader

In [ ]:
# PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY.
# IF NOT, THE AUTOGRADER WON'T KNOW WHO TO ASSIGN POINTS TO YOU IN OUR BACKEND

# YOUR PENN-ID GOES HERE AS AN INTEGER
STUDENT_ID =

In [ ]:
%%writefile notebook-config.yaml
grader_api_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/TestSandbox"
grader_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"
token_generator_url: "https://yj0vsrf8l4.execute-api.us-east-1.amazonaws.com/Initial/GetTokens"
token_generator_api_key: "dBB5ucuV5L6GyKFKmlrfP4bKkhtnA0bP42lR3rSh"

In [ ]:
grader = PennGrader('notebook-config.yaml', "cis5450_spr26_HW2", STUDENT_ID, STUDENT_ID, "cis5450_spr26")

We will use scores from Penn Grader to determine your grade. You will still need to submit your notebook so we can check for cheating and plagarism. Do not cheat.

**Note:** If you run Penn Grader after the due date for any question, your assignment will be marked late, even if you already had full points for the question before the deadline. To remedy this, if you're going to run your notebook after the deadline, either do not run the grading cells, or reinitialize the grader with an empty or clearly fake ID such as <code>999999999999</code> (please use 10+ digits to be clearly a fake <code>STUDENT_ID</code>)

# Indego Data

![POV: you've come across Harry on a sunny weekend](https://www.rideindego.com/wp-content/uploads/bb-plugin/cache/hero-classes-1024x682-landscape-8f85af3503a754b2a93e1c2636f14eaf-.jpg)



Bikeshare systems allow residents and visitors of a city to rent a bike for a short period of time to get around. Philadelphia has a bikeshare system called Indego; you may have seen one of the stations scattered around campus.

In this section of the assignment, you will run some SQL queries that allow you to analyze trip data shared by Indego: what parts of the city get the most trips, how does weather affect this, and so on.

We'll be parsing this data into dataframes and relations, and then exploring how to query and assemble the tables into results.

In [ ]:
# download and unpack data

!apt install zstd
!wget -nc -O hw2_data.tar.zst https://www.dropbox.com/scl/fi/h1hssju4jtga47mp4l2id/hw2_data.tar.zst?rlkey=q32a31i9e22lu4c7rgzvaifam&st=4iipkrgp
!tar xfv hw2_data.tar.zst
!mv hw2_data/* .
!rm -rf hw2_data/

## Part 1: Load & Process our Datasets [6 points total]

Before we get into the data, we first need to load and clean our datasets.

You'll be working with three CSV files:
- `stations.csv`, containing information about the location, name, and neighborhood of each station.
- `trips.csv`, containing detailed information about the kind, time, and locations of trips taken in Q4 of 2024.
- `weather.csv`, containing weather information for each day of September 2024 in Philadelphia.

**TODO**:
- Load `stations.csv` into a dataframe called `raw_stations_df`.
- Load `trips.csv` into a dataframe called `raw_trips_df`.
- Load `weather.csv` into a dataframe called `raw_weather_df`.

In [ ]:
# TODO: Import the datasets to pandas dataframes -- make sure the dataframes are named correctly!
raw_stations_df = pd.read_csv("stations.csv")
raw_stations_df

In [ ]:
raw_trips_df = pd.read_csv("trips.csv")
raw_trips_df

In [ ]:
raw_weather_df = pd.read_csv("weather.csv")
raw_weather_df

### 1.1 Data Preprocessing



#### 1.1.1 Cleaning the DataFrames [6 points]

You have a few tasks that you'll need to take care of to remove redundant information and make sure our rows match up nicely.
- The IDs used in the `stations_df` and `trips_df` are different. Transform the IDs in `stations_df` so that they follow the same format as the start/end station IDs you see in `trips_df`.
- Although we have trip data for several months, we will only do our analysis on trips started in September 2024. Remove all rows from `trips_df` that we will not need.
- Drop columns so that we have the following schemas:

**Final Schemas**:

For `stations_df`:
|station_id|station_name|latitude|longitude|neighborhood|
|---|---|---|---|---|

For `trips_df`:
|ride_id|duration|started_at|ended_at|start_station_id|end_station_id|trip_route_category|passholder_type|rideable_type|
|---|---|---|---|---|---|---|---|---|

For `weather_df`:
|date|temp_max_f|temp_min_f|temp_avg_f|rain_in|wind_speed_max_mph|
|---|---|---|---|---|---|

In [ ]:
stations_df =

In [ ]:
trips_df =

In [ ]:
weather_df =

In [ ]:
# 6 points
grader.grade(test_case_id = 'test_cleaning', answer = (trips_df.shape, trips_df.head(5), stations_df, weather_df))

## Part 2: Exploring the Data with DuckDB and SQL [98 points total]

### 👇👇👇 IMPORTANT: Pay VERY CLOSE attention to this style guide! If you ask for help on a query that is poorly formatted, you will be asked to reorganize your code before you get an answer.👇👇👇

The typical flow to use duckdb is:
1. Write a SQL query in the form of a string
  
  - **ALL SQL CLAUSES (`SELECT`, `FROM`, `WHERE`...) SHOULD BE CAPITALIZED!!!**. We would be checking the usage of certain clauses in your queries for partial credits and **they must be capitalized for you to receive credits**.
  
  - **String Syntax:** use triple quotes `"""<your query>"""` to write multi-line strings
  
  - **Aliases are your friend:** if there are very long table names or you find yourself needed to declare the source (common during join tasks), you should alias your tables with descriptive names
  
  - **New Clauses New Line:** each of the main SQL clauses (`SELECT`, `FROM`, `WHERE`, etc.) should begin on a new line
  
  - **Use Indentation:** if there are many components for a single clause, separate them out with new <ins>indented</ins> lines.

    Example below:
    ```SQL
    """
    SELECT ltn.some_id, SUM(stn.some_value) AS total
    FROM long_table_name AS ltn
         INNER JOIN short_table_name AS stn
            ON ltn.common_key = stn.common_key
         INNER JOIN med_table_name AS mtn
            ON ltn.other_key = mtn.other_key
    WHERE ltn.col1 > value
         AND stn.col2 <= another_value
         AND mtn.col3 != something_else
    GROUP BY ltn.some_id
    ORDER BY total
    """
    ```
2. Run the query using **duckdb.sql(your_query)**.

Duckdb allows you to reference the dataframes that are currently defined in your notebook, so you will be able to fully utilize the dataframes that you have created above!

Given that it is a brand new language, we wanted to give you a chance to directly compare the similarities/differences of the pandas that you already know and the SQL that you are about to learn. Please be patient as your queries might take a little while to run.

**⚠WARNING: DO NOT USE PANDAS FOR SQL QUESTIONS OR VICE-VERSA! OTHERWISE, YOU WON'T RECEIVE CREDITS FOR THESE QUESTIONS**

In [ ]:
#TODO: Run this cell to understand how Duck DB connects to local dataframes and queries them
test_duckdb_query = """
SELECT *
FROM trips_df
LIMIT 10
"""

duckdb.sql(test_duckdb_query).df()

### 2.1 User Engagement Analytics

In this part, you will be asked to complete the tasks in both pandas and sql.

## 2.1 Ridership Analytics

### 2.1.1 Counting Trips Per Station [10 points]

Analyze station popularity by counting the number of trips originating from each station.

**TODO:**
- For each station, **count the total number of trips** that started there using `trips` and `stations`.
- Select only stations with **more than 200 trips**.
- Display: `station_id`, `station_name`, `neighborhood`, and `total_trips`.
- **Order by** `total_trips` DESC, then `station_name` ASC.

Save your query as `query_station_trips` and result as `station_trips_df`.

**Final Schema:**

> station_id | station_name | neighborhood | total_trips

In [ ]:
duckdb.sql

In [ ]:
# TODO: Count the total number of trips per station

trips_per_station_df =

In [ ]:
# TODO: Count the total number of trips per station (SQL)
query_trips_per_station = '''
'''

# Execute query and save the result
trips_per_station_sql_df =

In [ ]:
# 10 points
grader.grade(test_case_id = 'test_trips_per_station', answer = (trips_per_station_df, trips_per_station_sql_df))

#### 2.1.2 Customer Type per Station [10 points]

There are two kinds of long-term passes: Indego30 and Indego365. All other passholder types are short-term.

For each station from **2.1.1** (stations with >200 trips), calculate:

  - **Average trip duration** in minutes
  - **Count of member trips** vs **casual trips**
  - **Order by** `avg_duration` DESC, then `station_name` ASC.

**Hint:** Use `COUNT(CASE WHEN ... THEN 1 END)` to count by rider type.

Save your query as `query_station_rider_type` and result as `station_rider_type_df`.**Final Schema:**

> station_id | station_name | avg_duration | long_passholders | short_passholders

In [ ]:
station_rider_type_df =


In [ ]:
query_station_rider_type = '''


'''

# Execute query and save the result
station_rider_type_sql_df =

In [ ]:
# 10 points
grader.grade(test_case_id = 'test_station_rider_type', answer = (station_rider_type_df, station_rider_type_sql_df))

#### 2.1.3 Average Trip Duraction by Neighborhood [10 points]

Compare ridership patterns across neighborhoods.**TODO:**
- Calculate the **average trip duration** for trips starting in each neighborhood.- Make sure to count **unique stations** in each neighborhood.- Display: `neighborhood`, `total_trips`, `unique_stations`, `avg_duration`.- **Order by** `avg_duration` DESC.

Save your query as `query_neighborhood_stats` and result as `neighborhood_stats_df`.**Final Schema:**

> neighborhood | total_trips | unique_stations | avg_duration

In [ ]:
neighborhood_stats_df =

In [ ]:
query_neighborhood_stats = '''


'''

# Execute query and save the result
neighborhood_stats_sql_df =

In [ ]:
# 6 points
grader.grade(test_case_id='test_neighborhood_stats', answer=(neighborhood_stats_df, neighborhood_stats_sql_df))

### 2.1.4 Neighborhood with Highest Trips Per Station [10 points]

Identify which neighborhood has the most efficient station utilization.

**TODO:**
- For each neighborhood, calculate:
  - Total trips  - Number of unique stations
  - **Trips per station** (rounded to 1 decimal)- Display: `neighborhood`, `total_trips`, `unique_stations`, `trips_per_station`.- **Order by** `trips_per_station` DESC.

**Hint:** Use `ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT ...), 1)` for the ratio.

Save your query as `query_trips_per_station` and result as `trips_per_station_df`.**Final Schema:**

> neighborhood | total_trips | unique_stations | trips_per_station

In [ ]:
trips_per_station_df =


In [ ]:
query_trips_per_station = '''



'''

trips_per_station_sql_df =

In [ ]:
# 10 points
grader.grade(test_case_id='test_trips_per_station_ratio', answer=(trips_per_station_df, trips_per_station_sql_df))

### 2.2 Temporal Patterns: Analyzing Trends Over Time

For the rest of the assignment, we will only be using **SQL**.

### 2.2.1 Hourly Ridership Patterns [5 points]

Analyze when riders are most active during the day.

**TODO:**
- Count the **total number of trips per hour** of the day.
- Calculate the **average trip duration** for each hour.
- Display: `hour`, `total_trips`, `avg_duration`.
- **Order by** `hour` ASC.

Save your query as `query_hourly_patterns` and result as `hourly_patterns_df`.

**Final Schema:**
> hour | total_trips | avg_duration

In [ ]:
# TODO: Calculate the total number of trips per hour
query_hourly_patterns = '''


'''

# Execute query and save the result
hourly_patterns_df =

In [ ]:
# 5 points
grader.grade(test_case_id='test_hourly_patterns', answer=hourly_patterns_df)

Next, **visualize** the trend using an appropriate choice of plot to identify hourly patterns for trips in September. Make sure to have the graph very clearly labeled. You can use a library of your choice.

Your graph should have:

* Clear title and axis labels (Hour of Day on X axis and Total Tips on Y axis)
* Adequate figure size to clearly show the trend
* Adjusted spacing to prevent any unseemly overlapping

You'll need to save your visualization as `"your_figure.png"`--with default pandas plotting, you'd probably find the `.savefig()` method helpful. Make sure to call that before any calls to `.show()`.

In [ ]:
# 3 points - LLM graded. If you are having trouble with the autograder,
# or would prefer a human grader for any reason, please make a private post
# on Ed with your answer and a TA will grade your work within 1 week.

import base64

with open("your_figure.png", "rb") as f:
  data = f.read()

encoded = base64.b64encode(data).decode("utf-8")

grader.grade(test_case_id='test_your_figure', answer=encoded)

**Your Observations:**

### 2.2.2 Daily Ridership with Rolling Average [7 points]

Analyze daily ridership trends with smoothing to identify patterns.

**TODO:**
- Calculate **total trips per day**.
- Compute a **3-day rolling average** of trips to smooth daily fluctuations.
- Round the rolling average to the nearest whole number.
- Display: `date`, `total_trips`, `rolling_avg_3day`.
- **Order by** `date` ASC.

**Hint:** Use window function `AVG(...) OVER (ORDER BY ... ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`.

Save your query as `query_daily_rolling` and result as `daily_rolling_df`.

**Final Schema:**
> date | total_trips | rolling_avg_3day

In [ ]:
query_daily_rolling = '''


'''

# Execute query and save the result
daily_rolling_df =

In [ ]:
# 7 points
grader.grade(test_case_id='test_daily_rolling', answer=daily_rolling_df)

### 2.2.3 Weekday vs Weekend Ridership [5 points]

Compare ridership patterns between weekdays and weekends.

**TODO:**
- Categorize each trip as `'Weekday'` or `'Weekend'` based on the day of the week that the trip started on.
- Calculate **total trips** and **average duration** for each category.
- Calculate the **percentage of long-term passholder rides** for each category.
- Display: `day_type`, `total_trips`, `avg_duration`, `longterm_pass_percentage`.
- **Order by** `total_trips` DESC.

**Hint:** Saturday and Sunday are weekends; use `CASE WHEN day_of_week IN ...`.

Save your query as `query_weekday_weekend` and result as `weekday_weekend_df`.

**Final Schema:**
> day_type | total_trips | avg_duration | longterm_pass_percentage

In [ ]:
query_weekday_weekend = '''


'''

weekday_weekend_df =

In [ ]:
grader.grade(test_case_id='test_weekday_weekend', answer=weekday_weekend_df)

### 2.3 Station Performance Analytics

### 2.3.1 Top Stations by Neighborhood [6 points]

Identify the busiest stations in each neighborhood.

**TODO:**
- Find the **top 3 stations in each neighborhood** by total trips.- Only include neighborhoods with at least three stations.- Display: `station_id`, `station_name`, `neighborhood`, `total_trips`.- **Order by** `neighborhood` ASC, `total_trips` DESC.

**Hint:** Use `ROW_NUMBER() OVER (PARTITION BY neighborhood ORDER BY ... DESC)` to rank stations within each neighborhood.

Save your query as `query_top_stations` and result as `top_stations_df`.**Final Schema:**

> station_id | station_name | neighborhood | total_trips

In [ ]:
query_top_station = '''


'''

top_stations_df =

In [ ]:
# 6 points
grader.grade(test_case_id='test_top_stations', answer=top_stations_df)

### 2.3.2 Station Performance vs Neighborhood Average [6 points]

Compare each station's performance to its neighborhood average.

**TODO:**
- For stations from **2.3.1** (top 3 per neighborhood), calculate:
  - Total trips for each station  - Average trips across all stations in the same neighborhood- Label stations as `'Above Average'` or `'Below Average'` based on comparison.- Display: `station_id`, `station_name`, `neighborhood`, `total_trips`, `neighborhood_avg`, `performance`.- **Order by** `performance` ASC, `total_trips` DESC, `station_id` ASC

Save your query as `query_station_performance` and result as `station_performance_df`.**Final Schema:**

> station_id | station_name | neighborhood | total_trips | neighborhood_avg | performance

In [ ]:
query_station_performance ='''


'''

station_performance_df =

In [ ]:
# 6 points
grader.grade(test_case_id='test_station_performance', answer=station_performance_df)

### 2.3.3 Daily Ridership Trends with LAG [6 points]

Analyze day-over-day changes in ridership.

**TODO:**
- For each day, calculate **total trips**.- Use `LAG()` to get the **previous day's trip count**.- Label each day's trend as `'Increasing'`, `'Decreasing'`, or `'Stable'` if the percentage change is between -3% and 3%.  - If no previous day data, label as `'N/A'`.- Display: `date`, `total_trips`, `prev_day_trips`, `trend`.- **Order by** `date` ASC.

Save your query as `query_daily_trends` and result as `daily_trends_df`.**Final Schema:**

> date | total_trips | prev_day_trips | trend

In [ ]:
# TODO: Analyze review trends over time
query_daily_trends = '''


'''

# Execute query and save the result
daily_trends_df =

In [ ]:
# 6 points
grader.grade(test_case_id='test_daily_trends', answer=daily_trends_df)

### 2.3.4 Inbound vs Outbound Trip Ratio [8 points]

Analyze station balance by comparing trips starting vs ending at each station.

**TODO:**
- For each station, calculate:

  - **Outbound trips** (trips starting at the station)

  - **Inbound trips** (trips ending at the station)

  - **Net flow** (outbound - inbound)

  - **Flow ratio** (outbound / inbound), rounded to 2 decimal places
- Handle division by zero using `CASE WHEN` or `NULLIF`.
- Label stations as `'Source'` (ratio > 1.1), `'Sink'` (ratio < 0.9), or `'Balanced'`.
- Only include stations with **more than 100 total trips** (inbound + outbound).
- Display: `station_id`, `station_name`, `outbound`, `inbound`, `net_flow`, `flow_ratio`, `station_type`.
- **Order by** `flow_ratio` DESC.
- **Limit to 15** results.

Save your query as `query_flow_analysis` and result as `flow_analysis_df`.

**Final Schema:**
> station_id | station_name | outbound | inbound | net_flow | flow_ratio | station_type

In [ ]:
query_flow_analysis = '''


'''

# Execute query and save the result
flow_analysis_df =

In [ ]:
# 8 points
grader.grade(test_case_id='test_flow_analysis', answer=flow_analysis_df)

### 2.3.5 Weather Impact on Ridership [5 points]

Analyze how weather affects bike ridership.

**TODO:**
- Join `trips` with `weather` on the date.
- Categorize days as `'Rainy'` (precipitation > 0 inches) or `'Dry'`.
- For each weather category, calculate:
  - Total trips
  - Average trip duration
  - Fraction of trips taken on e-bikes
  - Average temperature
- Display: `weather_type`, `total_trips`, `avg_duration`, `avg_temp`, `fraction_ebike`.
- **Order by** `total_trips` DESC.

Save your query as `query_weather_impact` and result as `weather_impact_df`.

**Final Schema:**
> weather_type | total_trips | avg_duration | avg_temp

In [ ]:
query_weather_impact = '''


'''

weather_impact_df =

In [ ]:
# 5 points
grader.grade(test_case_id='test_weather_impact', answer=weather_impact_df)

### 2.4 Peak Hour Analysis by Neighborhood [7 points]

Identify the busiest hour for each neighborhood.

**TODO:**
- For each neighborhood, find the **hour with the most trips**.- Calculate total trips during that peak hour.- Display: `neighborhood`, `peak_hour`, `peak_hour_trips`, `total_neighborhood_trips`, `peak_percentage`.- **Order by** `total_neighborhood_trips` DESC.

**Hint:** Use a CTE to calculate trips per neighborhood-hour, then use `ROW_NUMBER()` to find the peak.

Save your query as `query_peak_hours` and result as `peak_hours_df`.**Final Schema:**

> neighborhood | peak_hour | peak_hour_trips | total_neighborhood_trips | peak_percentage

In [ ]:
query_peak_hours = '''


'''

peak_hours_df =

In [ ]:
grader.grade(test_case_id="test_peak_hours", answer=peak_hours_df)

# Part 3: Schema Design

## 3.1 Functional Dependencies (5 points)


Let  $A, B, C, D, E, F$  all be attributes in our schema.

Use Armstrong's axioms to prove that if:

$AB \rightarrow C$, $C \rightarrow D$, $DE \rightarrow F$, and $E \rightarrow B$, then $AE \rightarrow F$.

Review Lecture 6 (Relational schema) slide 50 if you need a refresher on Armstrong's axioms.

In [ ]:
# 5 points - LLM graded. If you are having trouble with the autograder,
# or would prefer a human grader for any reason, please make a private post
# on Ed with your answer and a TA will grade your work within 1 week.

proof = """
<write your proof here -- use the style shown in class slides>
"""

grader.grade(test_case_id='test_armstrong_proof', answer=proof)

## 3.2 3NF (5 points)



**Please answer the following question in the markdown cell below.**

Suppose $R = {A, B, C, D, E}$, and we have the following set of functional dependencies:

$A \rightarrow BC$

$B \rightarrow C$

$B \rightarrow D$

$C \rightarrow E$

Decompose $R$ into third normal form (3NF). Clearly mark the relations in your final answer, and specify their primary key.

Note that a similar question to this has appeared on every 5450 midterm for the last 3 years.

In [ ]:
# 5 points - LLM graded. If you are having trouble with the autograder,
# or would prefer a human grader for any reason, please make a private post
# on Ed with your answer and a TA will grade your work within 1 week.

decomp = """
<write your decomposition here -- only the final answer is required>
"""

grader.grade(test_case_id='test_3nf_decomp', answer=decomp)

## 3.3 Entity Relationship Design (10 points)


Please answer the following question by uploading an image of your solution. **The image must be a PNG.** You can do this by:


1. Upload images from your local machine by clicking on the files icon in the left sidebar
2. Click the **Upload** button
3. Select the image you want to upload after the file upload dialogue appears

Consider a university in which students can enroll in classes, each class can be taught by one or more faculty (and faculty may teach multiple courses), and each faculty is assigned a single admin. To support its application, the schema must contain student and faculty demographic data, course information such as course number and description, and admin contact info.

Design an ER diagram that captures the relationships above. Be sure to capture all entities, relationships, and cardinalities.

In [ ]:
# 10 points - LLM graded. If you are having trouble with the autograder,
# or would prefer a human grader for any reason, please make a private post
# on Ed with your answer and a TA will grade your work within 1 week.

import base64

with open("your_er_diagram.png", "rb") as f:
  data = f.read()

encoded = base64.b64encode(data).decode("utf-8")

grader.grade(test_case_id='test_er_diagram', answer=encoded)

# HW Submission

Before you submit on Gradescope (you must submit your notebook to receive credit):


1.   Restart and Run-All to make sure there's nothing wrong with your notebook
2.   **Double check that you have the correct PennID (all numbers) in the autograder**.
3. Make sure you've run all the PennGrader cells
4. Go to the "File" tab at the top left, and download both the .ipynb and .py files, renaming them as "homework2.ipynb" and "homework2.py" respectively. Upload both files to Gradescope directly!


**You MUST verify that the autograder finishes running and gives you your expected score (not a 0).**

**Let the course staff know ASAP if you have any issues submitting, but otherwise best of luck! Congrats on finishing the HW.**